In [7]:
import pandas as pd
from statsforecast.models import MSTL
from statsforecast.models import MSTL
import pandas as pd
from google.cloud import bigquery

In [8]:
client = bigquery.Client()

/mnt/encrypted_data/git/data-testing/venv/lib/python3.9/site-packages/google/auth/_default.py:78: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/mnt/encrypted_data/git/data-testing/venv/lib/python3.9/site-packages/google/auth/_default.py:78: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [15]:
EXECUTION_DATE = "2023-01-01"
current_forecast_config = {
    "source_dataset": "pipe_ais_v3_alpha_published",
    "source_table": "stats_daily",
    "source_date_column_sql": "date",
    "source_forecast_column_sql": "MAX(raw_positions)",
    "source_sql": None,
    "algorithm": "mstl",
    "algorithm_parameters": {"season_length": [365, 7]},
    "train_start": "2012-01-01",
    "train_end": EXECUTION_DATE,
    "forecast_periods": 1,
    "target_dataset": "scratch_christian_homberg_ttl120d",
    "target_table": "anomaly_detection_forecasts"
}

In [17]:
algorithm_parameters

{'season_length': [365, 7]}

In [16]:
for key, value in current_forecast_config.items():
    globals()[key] = value

In [23]:
if (source_sql is not None):
    df_timeseries = pd.read_gbq(source_sql)
else:
    df_timeseries = pd.read_gbq(f'''
    SELECT 
        {source_date_column_sql} date, 
        {source_forecast_column_sql} y
    FROM {source_dataset}.{source_table}
    WHERE {source_date_column_sql} BETWEEN '{train_start}' AND '{train_end}'
    GROUP BY date
    ORDER BY date
    ''')

np_timeseries = df_timeseries["y"].to_numpy().astype(int)

In [24]:
mstl_model = MSTL(**algorithm_parameters)

In [25]:
mstl_model.forecast(np_timeseries, 1)

{'mean': array([88152215.10579823])}